# 99. Label Studio project 안전 삭제

이 notebook은 대량 task가 있는 project를 Python SDK/REST API로 삭제할 때 사용합니다. 위험도가 높기 때문에 번호를 `99`로 두고, 기본값은 preview만 하도록 설정했습니다.

삭제는 두 단계입니다.
1. task를 batch 단위로 삭제합니다.
2. 선택적으로 project meta를 삭제합니다.

안전장치:
- `DRY_RUN=True`가 기본값입니다.
- 실제 삭제하려면 `DRY_RUN=False`와 `CONFIRM="delete"`를 둘 다 설정해야 합니다.
- 처음에는 `MAX_BATCHES=1`로 한 batch만 테스트하는 것이 안전합니다.


In [ ]:
from pathlib import Path
import sys

# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from labelstudio_bbox_tools.config import settings_from_env

settings = settings_from_env(REPO_ROOT / ".env")
print("repo:", REPO_ROOT)
print("Label Studio URL:", settings.url)
print("doc_root:", settings.doc_root)


In [ ]:
from labelstudio_bbox_tools.admin.project_delete import inspect_project, delete_project_safely

PROJECT_ID = 0
PAGE_SIZE = 1000
BATCH_SIZE = 1000
SLEEP_SEC = 0.15

# 실제 project meta까지 삭제하려면 True로 바꿉니다.
DELETE_PROJECT_META = False

# 실제 삭제를 하려면 아래 두 조건을 모두 만족해야 합니다.
DRY_RUN = True
CONFIRM = ""  # 실제 삭제 시에만 "delete" 입력

# 처음 실제 실행할 때는 1로 두고 한 batch만 지워지는지 확인하는 것을 권장합니다.
# 전체 삭제하려면 None으로 바꿉니다.
MAX_BATCHES = 1


In [ ]:
inspect_project(settings.url, settings.api_key, PROJECT_ID)


In [ ]:
summary = delete_project_safely(
    ls_url=settings.url,
    api_key=settings.api_key,
    project_id=PROJECT_ID,
    page_size=PAGE_SIZE,
    batch_size=BATCH_SIZE,
    sleep_sec=SLEEP_SEC,
    delete_project_meta=DELETE_PROJECT_META,
    dry_run=DRY_RUN,
    confirm=CONFIRM,
    max_batches=MAX_BATCHES,
)
summary.as_dict()
